# Similarity Algorithms in Neo4j

<a target="_blank" href="https://colab.research.google.com/github/neo4j/graph-data-science-client/blob/main/examples/similarity_algorithms.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


This Jupyter notebook is inspired by examples in the [Neo4j Graph Data Science Client Github repository](https://github.com/neo4j/graph-data-science-client/tree/main/examples).

The notebook demonstrates the usage of the `graphdatascience` library for performing similarity analysis on the **SNAP MOOC dataset**, which can be downloaded [here](https://snap.stanford.edu/data/act-mooc.html).

The tasks covered here include data ingestion, structural node similarity (Jaccard, Overlap), and feature-based similarity using the K-Nearest Neighbors (KNN) algorithm.


## Dataset Overview: `act-mooc`

The **`act-mooc`** dataset (from Stanford's SNAP library) tracks user behavior on a Massive Open Online Course platform. It provides a real-world benchmark for graph-based modeling and analytics.

| Name | Type | Nodes | Edges | Description |
| :--- | :--- | :--- | :--- | :--- |
| **`act-mooc`** | Bipartite, Directed, Attributed, Temporal | 7,143 | 411,749 | Student actions on a MOOC platform, with binary drop-out labels. |

---

## Key Characteristics

* **Bipartite Structure:** Nodes are split into **Users** (~7,047 students) and **Targets** (97 course items).
* **Temporal & Attributed:** Every action (edge) includes precise timestamps and **4D feature vectors**.
* **Machine Learning Labels:** Tracks ground-truth student dropout status.

---

# Graph Similarity Algorithms in Neo4j GDS

This notebook demonstrates how to calculate and compare different graph similarity metrics using the SNAP `act-mooc` dataset and Neo4j Graph Data Science (GDS).

### Learning Objectives:
1.  **Data Ingestion**: Load large-scale interaction data into Neo4j Aura.
2.  **Graph Projections**: Create in-memory graphs for analysis.
3.  **Similarity Algorithms**: Execute Node Similarity in various modes (Stream, Stats, Mutate).
4.  **Metric Comparison**: Understand the differences between Jaccard, Overlap, Cosine similarities.

## Setup for SNAP `act-mooc` Dataset

In [ ]:
import os, requests

url = "https://snap.stanford.edu/data/act-mooc.tar.gz"
dest = "act-mooc.tar.gz"

print("Downloading and extracting dataset...")
with open(dest, "wb") as f:
    f.write(requests.get(url).content)

!mkdir -p act-mooc_data && tar -xzf {dest} -C act-mooc_data
print("Done. Contents:")
!ls -R act-mooc_data

### Data Extraction Complete
The MOOC interaction data (actions, features, and labels) has been successfully downloaded and extracted. We are now ready to begin the Neo4j ingestion process.

# Neo4j setup

In [ ]:
pip install neo4j

## Section 1: Data Preparation

In this section, we download the dataset and define the ingestion logic to map TSV files to a Graph Schema consisting of `MoocUser`, `MoocAction`, and `MoocActivity` nodes.

Get the Client ID and Client Secret from here: https://console.neo4j.io/account/api-keys

Refrence article: https://neo4j.com/docs/aura/api/authentication/

In [ ]:
from neo4j import GraphDatabase

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = "<URI>"
AUTH = ("<USERNAME>", "<PASSWORD>")
USERNAME = "<USERNAME>"
PASSWORD = "<PASSWORD>"

In [ ]:
CLIENT_ID = "<CLIENT_ID>"
CLIENT_SECRET = "<CLIENT_SECRET>"

### 1.1 Graph Data Science Setup

In [ ]:
pip install graphdatascience

In [ ]:
from graphdatascience.session import GdsSessions, AuraAPICredentials

PROJECT_ID = None
# Create a new GdsSessions object
sessions = GdsSessions(api_credentials=AuraAPICredentials(CLIENT_ID, CLIENT_SECRET, PROJECT_ID))

### 1.2 Connecting to Session

In [ ]:
from graphdatascience.session import DbmsConnectionInfo, SessionMemory

db_connection = DbmsConnectionInfo(
     uri=URI,
     username=USERNAME,
     password=PASSWORD
)

gds = sessions.get_or_create(
    session_name="Similarity",
    memory=SessionMemory.m_2GB,
    db_connection=db_connection,
)

### 1.3 Populate Neo4j Instance and Project Graph
This section creates a sample social network and projects it into the GDS session for analysis.

We will use [CONSTRAINTS](https://neo4j.com/docs/cypher-manual/current/schema/constraints/create-constraints/) here,

In Cypher, constraints are schema-level rules applied to node labels or relationship types to enforce data integrity, maintain consistency, and safeguard the graph.

In [ ]:
# Step 1: Tutorial Setup & Constants
from __future__ import annotations
import csv, math, os, sys
from decimal import Decimal
from pathlib import Path
from typing import Iterable, Iterator
from itertools import islice

# dataset paths
DATA_DIR = Path("/content/act-mooc_data/act-mooc")
DATASET_ID = "act-mooc"

# Schema constraints to ensure data integrity
CONSTRAINTS = [
    "CREATE CONSTRAINT mooc_user_identity IF NOT EXISTS FOR (u:MoocUser) REQUIRE (u.datasetId, u.userId) IS UNIQUE",
    "CREATE CONSTRAINT mooc_activity_identity IF NOT EXISTS FOR (t:MoocActivity) REQUIRE (t.datasetId, t.targetId) IS UNIQUE",
    "CREATE CONSTRAINT mooc_action_identity IF NOT EXISTS FOR (a:MoocAction) REQUIRE (a.datasetId, a.actionId) IS UNIQUE",
    "CREATE RANGE INDEX mooc_action_time IF NOT EXISTS FOR (a:MoocAction) ON (a.datasetId, a.timestampSeconds)"
]

### Step 2: The Ingestion Engine

In this step, we define the core functions responsible for reading the TSV files and batching the data. Using a generator (`yield`) and a chunking function helps us stay within memory limits and handle large datasets efficiently.

Note: We are limiting this to 150000 as free tier supports 2gb Memory, you can change this in session creation

*Modify 'memory=SessionMemory.m_2GB' per your needs*


In [ ]:
# Step 2: Define the Ingestion Engine
def get_action_rows(limit=150000):
    """Parses the 3 TSV files into a unified dictionary format."""
    a_path, f_path, l_path = DATA_DIR/"mooc_actions.tsv", DATA_DIR/"mooc_action_features.tsv", DATA_DIR/"mooc_action_labels.tsv"
    with open(a_path) as ah, open(f_path) as fh, open(l_path) as lh:
        zipped = zip(csv.DictReader(ah, delimiter='\t'), csv.DictReader(fh, delimiter='\t'), csv.DictReader(lh, delimiter='\t'))
        for i, (a_row, f_row, l_row) in enumerate(islice(zipped, limit)):
            yield {
                "actionId": i, "userId": int(a_row["USERID"]), "targetId": int(a_row["TARGETID"]),
                "timestampSeconds": int(Decimal(a_row["TIMESTAMP"])),
                "featureVector": [float(f_row[f"FEATURE{j}"]) for j in range(4)],
                "dropoutAfterAction": bool(int(l_row["LABEL"])),
                "sourceLabelActionId": int(l_row["ACTIONID"]),
            }

def chunked(rows, size=5000):
    batch = []
    for r in rows:
        batch.append(r)
        if len(batch) == size: yield batch; batch = []
    if batch: yield batch

### Step 3: Executing the Ingestion

Finally, we apply the database constraints and execute the batch import. We use `my_session.run_cypher` to send chunks of data to Neo4j. This approach ensures high throughput while preventing the transaction log from growing too large.

#### Import act-mooc Dataset
This cell will execute the import process to write the MOOC data to your Neo4j database. We pass `['import', '--write']` to the main function to trigger the writing logic.

In [ ]:
IMPORT_CYPHER = """
UNWIND $rows AS row
MERGE (u:MoocUser {datasetId: $datasetId, userId: row.userId})
MERGE (t:MoocActivity {datasetId: $datasetId, targetId: row.targetId})
MERGE (a:MoocAction {datasetId: $datasetId, actionId: row.actionId})
SET a.timestampSeconds = row.timestampSeconds,
    a.featureVector = row.featureVector,
    a.dropoutAfterAction = row.dropoutAfterAction
MERGE (u)-[:PERFORMED]->(a)
MERGE (a)-[:ON_ACTIVITY]->(t)
"""

# Step 3: Run the Ingestion
print("Applying schema constraints...")
for cmd in CONSTRAINTS: gds.run_cypher(cmd)

print("Starting batch import...")
total = 0
for batch in chunked(get_action_rows()):
    res = gds.run_cypher(IMPORT_CYPHER, params={"datasetId": DATASET_ID, "rows": batch})
    total += len(batch)
    if total % 25000 == 0: print(f"Imported {total} actions...")

print(f"\nSuccess! Total actions imported: {total}")

In [ ]:
import pandas as pd

# Query to count nodes by label for the 'act-mooc' dataset
verify_query = """
MATCH (n)
WHERE n.datasetId = 'act-mooc'
RETURN labels(n)[0] AS Label, count(n) AS Count
ORDER BY Count DESC
"""

# Using the existing GDS session to run the query
counts_df = gds.run_cypher(verify_query)
if counts_df.empty:
    print("No nodes found for dataset 'act-mooc'.")
else:
    display(counts_df)

#### Run Cypher Query on GDS Session
Use the cell below to execute Cypher queries. Replace the `query` string with your desired Cypher code.

In [ ]:
# Define your Cypher query here
query = """
MATCH (n)
RETURN labels(n) AS Label, count(*) AS Count
LIMIT 10
"""

# Execute the query using the active GDS session
result = gds.run_cypher(query)

# Display the results
display(result)

#### Database Stats: Labels and Relationships

In [ ]:
# Check Node Label counts
node_counts = gds.run_cypher("""
    MATCH (n)
    RETURN labels(n) AS Labels, count(*) AS Count
""")

# Check Relationship Type counts
rel_counts = gds.run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS Type, count(*) AS Count
""")


In [ ]:
print("--- Node Label Counts ---")
if node_counts.empty:
    print("No nodes found.")
else:
    display(node_counts)



In [ ]:
print("\n--- Relationship Type Counts ---")
if rel_counts.empty:
    print("No relationships found.")
else:
    display(rel_counts)

## Node Similarity Algorithm
We will now use the Graph Data Science Node Similarity algorithm. This algorithm compares sets of nodes (e.g., users) based on the nodes they are connected to (e.g., activities) using metrics like Jaccard Similarity.

## Section 2: Structural Node Similarity

Node Similarity in GDS compares sets of neighbors. Here, we compare users based on the activities they have interacted with. We will explore how different metrics respond to the graph structure.

In [ ]:
REMOTE_PROJECT_CYPHER = """
MATCH (u:MoocUser {datasetId: $datasetId})-[:PERFORMED]->(a:MoocAction)-[:ON_ACTIVITY]->(t:MoocActivity {datasetId: $datasetId})
RETURN gds.graph.project.remote(
  u,
  t,
  {
    sourceNodeLabels: labels(u),
    targetNodeLabels: labels(t),
    relationshipType: 'ACTED_ON'
  }
)
""".strip()

### 2.1 Project a Bipartite Graph for Node Similarity


In [ ]:
graph_name = 'user-activity-similarity'

# Drop graph if it already exists in the catalog
if gds.graph.exists(graph_name)['exists']:
    g_existing = gds.graph.get(graph_name)
    gds.graph.drop(g_existing)

# Execute the AGA remote projection via gds.graph.project
g_sim, project_result = gds.graph.project(
    graph_name,
    REMOTE_PROJECT_CYPHER,
    query_parameters={"datasetId": "act-mooc"}
)

print(f"Graph '{graph_name}' projected successfully.")
display(project_result)

#### Run Node Similarity in Stream mode

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Running Node Similarity algorithm...")
similarity_results = gds.nodeSimilarity.stream(
    g_sim,
    topK=5,
    similarityCutoff=0.1
)

if similarity_results.empty:
    print("No similarity pairs found with the current threshold (0.1).")
else:
    similarity_results = similarity_results.rename(columns={
        "node1": "user1_id",
        "node2": "user2_id",
        "similarity": "jaccard_score"
    })

    similarity_results['user1_id'] = similarity_results['user1_id'].astype(int)
    similarity_results['user2_id'] = similarity_results['user2_id'].astype(int)



### Advanced Node Similarity: Stats, Mutate, and Write Modes
Following the GDS documentation, we will now explore the other execution modes and result limiting techniques.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of Jaccard Similarity scores
plt.figure(figsize=(10, 6))
sns.histplot(similarity_results['jaccard_score'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Jaccard Similarity Scores between Users')
plt.xlabel('Jaccard Similarity Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 2.2 Mutate Mode: 
Save similarity scores as relationships within the in-memory graph
Mutate mode adds relationships to the projected graph and returns summary execution statistics.

In [ ]:

mutate_results = gds.nodeSimilarity.mutate(
    g_sim,
    mutateRelationshipType='SIMILAR_TO',
    mutateProperty='score',
    similarityCutoff=0.5 # Only highly similar pairs
)

print("Mutate Mode Results (Relationships added to in-memory graph):")
display(mutate_results)

In [ ]:
print(f"Nodes compared:        {mutate_results['nodesCompared']}")
print(f"Relationships created: {mutate_results['relationshipsWritten']}")
print(f"Compute time (ms):     {mutate_results['computeMillis']}")
print("\nSimilarity distribution:")
display(pd.Series(mutate_results['similarityDistribution']))

### 2.3 Write Mode: 
Persist the top similarity results back to the Neo4j Database
We write back the 'SIMILAR_TO' relationships and 'score' property mutated in Step 4.

In [ ]:


try:
    write_results = gds.graph.relationshipProperties.write(
        g_sim,
        'SIMILAR_TO',
        ['score']
    )
    print("Write Mode Results (Persisted mutated relationships back to Neo4j):")
    display(write_results)
except Exception as e:
    print(f"Write mode failed: {e}")

### Implementing Multiple Similarity Metrics
We will now compare the results of **Jaccard**, **Overlap**, and **Cosine** similarity metrics.

### Jaccard Similarity (Intersection / Union)
Best suited for comparing unweighted neighbour sets

In [ ]:

# Drop graph if it already exists in the catalog
gds.graph.drop('user-activity-similarity', failIfMissing=False)

# Project graph using AGA Remote Projection syntax
g_sim, project_result = gds.graph.project(
    'user-activity-similarity',
    REMOTE_PROJECT_CYPHER,
    query_parameters={"datasetId": "act-mooc"}
)

# Stream Jaccard Similarity results
jaccard_df = gds.nodeSimilarity.stream(
    g_sim,
    similarityMetric='JACCARD',
    topK=5,
    similarityCutoff=0.1
)

print("Top Pairs using Jaccard Similarity (Default):")
display(jaccard_df.head(5))

### Overlap Coefficient (Intersection / Min(|A|, |B|))
Best suited for detecting when one neighbour set is mostly contained in another

In [ ]:
# Drop graph if it already exists
gds.graph.drop('user-activity-similarity', failIfMissing=False)

# Project graph using AGA Remote Projection syntax
g_sim, project_result = gds.graph.project(
    'user-activity-similarity',
    REMOTE_PROJECT_CYPHER,
    query_parameters={"datasetId": "act-mooc"}
)

# Stream Overlap Coefficient results
overlap_df = gds.nodeSimilarity.stream(
    g_sim,
    similarityMetric='OVERLAP',
    topK=5,
    similarityCutoff=0.1
)

print("Top Pairs using Overlap Coefficient:")
display(overlap_df.head(5))

For **Cosine Similarity**, we need a relationship property to act as a weight. We will re-project the graph to count the number of actions between a User and an Activity as a weight.

### Cosine Similarity (Weighted Dot Product)
Best suited for weighted relationships

In [ ]:
# Drop graph if it already exists
gds.graph.drop('weighted-sim', failIfMissing=False)

# AGA Remote Projection for Weighted Graph
weighted_project_query = """
MATCH (u:MoocUser {datasetId: $datasetId})-[:PERFORMED]->(a:MoocAction)-[:ON_ACTIVITY]->(t:MoocActivity {datasetId: $datasetId})
WITH u, t, count(a) as interaction_count
RETURN gds.graph.project.remote(
  u,
  t,
  {
    sourceNodeLabels: labels(u),
    targetNodeLabels: labels(t),
    relationshipType: 'ACTED_ON',
    relationshipProperties: { weight: toFloat(interaction_count) }
  }
)
""".strip()

# Execute remote weighted graph projection
g_weighted, _ = gds.graph.project('weighted-sim', weighted_project_query, query_parameters={"datasetId": "act-mooc"})

# Stream Cosine Similarity using interaction_count weights
cosine_df = gds.nodeSimilarity.stream(
    g_weighted,
    similarityMetric='COSINE',
    relationshipWeightProperty='weight',
    topK=5,
    similarityCutoff=0.1
)

print("Top Pairs using Cosine Similarity (Weighted):")
display(cosine_df.head(5))

### Comprehensive Similarity Metric Comparison

We will now use Neo4j GDS similarity functions to compare users based on their aggregated activity feature vectors. We will calculate:
1. **Jaccard Similarity**: Measures the intersection over union.
2. **Overlap Similarity**: Measures the intersection over the size of the smaller set.
3. **Cosine Similarity**: Measures the cosine of the angle between vectors.
4. **Pearson Similarity**: Measures the linear correlation between vectors.

## Section 3: Feature-Based Vector Similarity

Beyond just looking at 'who clicked what', we can compare users based on the **content** of their actions. We aggregate the 4-dimensional feature vectors from each user's actions and compare these profile vectors using Cypher functions.

### Feature-Based Similarity: Averaging Features per User

We aggregate (average) the 4D feature vectors of all actions performed by a user to create a single representative "behavioral profile" vector for that user.
This profile captures their overall interaction pattern across activities.

In [ ]:
similarity_comp_query = """
MATCH (u:MoocUser {datasetId: 'act-mooc'})
WHERE u.userId IN [0, 1] // Comparing User 0 and User 1
MATCH (u)-[:PERFORMED]->(a:MoocAction)
WITH u.userId AS userId,
     [avg(a.featureVector[0]), avg(a.featureVector[1]), avg(a.featureVector[2]), avg(a.featureVector[3])] AS vector
WITH max(CASE WHEN userId = 0 THEN vector END) AS v1,
     max(CASE WHEN userId = 1 THEN vector END) AS v2
RETURN
    gds.similarity.jaccard(v1, v2) AS jaccard,
    gds.similarity.overlap(v1, v2) AS overlap,
    gds.similarity.cosine(v1, v2) AS cosine,
    gds.similarity.pearson(v1, v2) AS pearson
"""

sim_comparison = gds.run_cypher(similarity_comp_query)
display(sim_comparison)

### Similarity Comparison: User 2 vs User 3
We will now apply the same logic to compare the feature vectors of User 2 and User 3.

In [ ]:
similarity_comp_query_2 = """
MATCH (u:MoocUser {datasetId: 'act-mooc'})
WHERE u.userId IN [2, 3]
MATCH (u)-[:PERFORMED]->(a:MoocAction)
WITH u.userId AS userId,
     [avg(a.featureVector[0]), avg(a.featureVector[1]), avg(a.featureVector[2]), avg(a.featureVector[3])] AS vector
WITH max(CASE WHEN userId = 2 THEN vector END) AS v1,
     max(CASE WHEN userId = 3 THEN vector END) AS v2
RETURN
    gds.similarity.jaccard(v1, v2) AS jaccard,
    gds.similarity.overlap(v1, v2) AS overlap,
    gds.similarity.cosine(v1, v2) AS cosine,
    gds.similarity.pearson(v1, v2) AS pearson
"""

sim_comparison_2 = gds.run_cypher(similarity_comp_query_2)
display(sim_comparison_2)

## Section 4: K-Nearest Neighbors (KNN)

While Node Similarity uses the graph structure (shared activities), **KNN** finds similar nodes based on specific properties—like the feature vectors we've been looking at.

We will:
1.  **Aggregate Features**: Average the action features for each user.
2.  **Project Node Properties**: Create a graph projection where `MoocUser` nodes have a `featureVector` property.
3.  **Run KNN**: Find the top-K most similar users based on Euclidean distance of those vectors.

### 4.1 Prepare User Feature Vectors in the Database


#### Feature-Based Profile Aggregation:
We calculate the ***average*** across all 4 interaction features for each user's actions.
This condenses a user's multi-action history into a single 4D behavioral profile vector
stored on the MoocUser node, enabling property-based similarity algorithms like KNN.


In [ ]:
print("Calculating average feature vectors for users...")


gds.run_cypher("""
MATCH (u:MoocUser {datasetId: 'act-mooc'})-[:PERFORMED]->(a:MoocAction)
WITH u,
     avg(a.featureVector[0]) as f1,
     avg(a.featureVector[1]) as f2,
     avg(a.featureVector[2]) as f3,
     avg(a.featureVector[3]) as f4
SET u.featureVector = [f1, f2, f3, f4]
""")

### 4.2 Project Graph with Properties
Remote projection including the node property


In [ ]:
knn_graph_name = 'user-features-knn'

# Single-line graph drop
gds.graph.drop(knn_graph_name, failIfMissing=False)

# Remote KNN Graph Projection string for AGA setup
REMOTE_KNN_PROJECT = """
MATCH (u:MoocUser {datasetId: $datasetId})
RETURN gds.graph.project.remote(
  u,
  null,
  {
    sourceNodeLabels: labels(u),
    sourceNodeProperties: { featureVector: u.featureVector }
  }
)
""".strip()

# Execute project remotely via standard gds.graph.project
g_knn, _ = gds.graph.project(
    knn_graph_name,
    REMOTE_KNN_PROJECT,
    query_parameters={"datasetId": "act-mooc"}
)

### 4.3 Run KNN Stream


In [ ]:
print("Running KNN algorithm...")

# Execute KNN stream and display the top results directly
knn_results = gds.knn.stream(
    g_knn,
    nodeProperties=['featureVector'],
    topK=3,
    sampleRate=1.0,
    randomSeed=42
)

# Rename columns for clean display
knn_results = knn_results.rename(columns={
    "node1": "user_a",
    "node2": "user_b",
    "similarity": "score"
})

print("Top KNN Similarity results (Feature-based similarity):")
display(knn_results.sort_values('score', ascending=False).head(10))

###  Summary of top 5 most similar user pairs from KNN results


In [ ]:
top_5_knn = knn_results.sort_values('score', ascending=False).head(5)

print("Top 5 Most Similar User Pairs (KNN):")
for index, row in top_5_knn.iterrows():
    print(f"- User {int(row['user_a'])} and User {int(row['user_b'])} with a similarity score of {row['score']:.4f}")

display(top_5_knn)

### Comparing KNN (Behavioral) vs. Jaccard (Structural) Similarity

In this step, we merge the results from the KNN algorithm (which used averaged feature vectors) with the Jaccard similarity results (which used shared activity neighbors). This helps us identify if users who *act* similarly also interact with the *same* activities.

Ensure column names are consistent for merging

jaccard_df has [node1, node2, similarity]

knn_results has [user_a, user_b, score]

Rename jaccard_df for a clean merge

In [ ]:
import pandas as pd

# Prepare Jaccard results for comparison
jaccard_compare = jaccard_df.rename(columns={
    'node1': 'user_a',
    'node2': 'user_b',
    'similarity': 'jaccard_score'
})

# Convert IDs to integer to ensure matching types across datasets
jaccard_compare['user_a'] = jaccard_compare['user_a'].astype(int)
jaccard_compare['user_b'] = jaccard_compare['user_b'].astype(int)
knn_results['user_a'] = knn_results['user_a'].astype(int)
knn_results['user_b'] = knn_results['user_b'].astype(int)

# Inner merge on matching user pairs
comparison_df = pd.merge(
    knn_results,
    jaccard_compare,
    on=['user_a', 'user_b'],
    how='inner'
)

print(f"Found {len(comparison_df)} pairs that appear in both similarity results.")

# Display top overlapping similarity comparisons
if not comparison_df.empty:
    display(comparison_df.sort_values(by=['score', 'jaccard_score'], ascending=False).head(10))
else:
    print("No overlapping pairs found between KNN and Jaccard sets at the current thresholds.")

## Final Summary: Multi-Faceted Similarity Analysis

In this notebook, we successfully built an end-to-end Graph Data Science pipeline using the **SNAP `act-mooc`** dataset and **Neo4j Aura**.

### Key Stages Completed:
1. **Data Ingestion**: We managed a high-volume import of 150,000 actions, mapping users and activities into a structured graph schema while respecting Neo4j Aura's free-tier limits.
2. **Structural Similarity (Jaccard & Overlap)**: We identified users who interact with the same resources. Our comparison showed that many users share nearly identical activity sets, which is ideal for collaborative filtering recommendations.
3. **Feature-Based Similarity (KNN)**: We used behavior-based vectors (averages of action features) to find users who *act* similarly, even if they don't share the same target activities.
4. **Metric Comparison**: We successfully merged structural and behavioral results, finding 116 user pairs that were highly similar in both metrics—proving that behavioral profiles are strong indicators of shared intent in this dataset.

### Which Metric to Use?
- **Jaccard**: Best for 'people who bought this also bought that' scenarios.
- **Overlap**: Best for detecting 'Expert' vs. 'Novice' users where one user's history is a subset of another's.
- **KNN (Cosine/Euclidean)**: Best for content-based matching based on latent features or interaction style.